# Notebook 02 - Analisis Fiscal Argentina 2020-2026

**Fuente:** Secretaria de Hacienda | AIF (Sector Publico Base Caja) + IMIG  
**Deflactor:** IPC Nivel General INDEC  
**Unidad:** pesos constantes del ultimo mes disponible

**Alcance titulares:** Sector Publico Total = Adm. Nacional + PAMI + Fondos Fiduciarios  
**Periodo de ajuste:** diciembre 2023 (asuncion Milei) → ultimo mes disponible

### Graficos generados
1. Resultado Primario y Financiero (serie mensual 2020-2026)
2. Composicion del gasto corriente (mensual)
3. Composicion de ingresos (mensual)
4. Transferencias a provincias (mensual)
5. Ajuste por componente AIF (variacion dic-23 → ultimo mes)
6. Torta: % de la baja del gasto por rubro (dic-23 → ultimo mes)
7. Barras: variacion real por rubro (dic-23 → ultimo mes)

In [ ]:
# ── Celda 1: Dependencias y helpers ──────────────────────────────────
import sys, io, os, zipfile, requests
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q pandas matplotlib openpyxl

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DPI      = 600
FIG_W    = 26
FIG_H    = 7
GRAFICOS = []
GAP_DATE = pd.Timestamp('2022-06-01')

# Milei asumio diciembre 2023 → referencia del ajuste
MES_INICIO = pd.Timestamp('2023-12-01')

plt.rcParams['figure.dpi']        = 100
plt.rcParams['savefig.dpi']       = DPI
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size']         = 9

MESES_ES = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
            7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}

def drop_gap(serie):
    return serie.drop(GAP_DATE, errors='ignore')

def fmt_int(ax, idx):
    """Eje X con etiquetas trimestrales en español sobre posiciones enteras."""
    ticks, labels = [], []
    for i, d in enumerate(idx):
        if d.month in [1, 4, 7, 10]:
            ticks.append(i)
            labels.append(f"{MESES_ES[d.month]}-{d.year}")
    ax.set_xticks(ticks)
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
    ax.set_xlim(-0.8, len(idx) - 0.2)

def save_fig(fig, nombre):
    fname = f'{nombre}.png'
    fig.savefig(fname, dpi=DPI, bbox_inches='tight')
    GRAFICOS.append(fname)
    print(f'  Guardado: {fname}')

print('OK')

In [ ]:
# ── Celda 3: Grafico 01 - Resultado Primario y Financiero ─────────────
primario   = drop_gap(get_serie_total('XIV_RESULTADO_PRIMARIO'))
financiero = drop_gap(get_serie_total('XV_RESULTADO_FINANCIERO'))
prim_real  = deflactar(primario)
fin_real   = deflactar(financiero)
idx = primario.index
pos = list(range(len(idx)))\n\nfig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
colors = ['#27ae60' if v >= 0 else '#c0392b' for v in prim_real]
ax.bar(pos, prim_real / 1e6, color=colors, alpha=0.9, label='Resultado Primario', zorder=2)
ax.plot([p + 0.5 for p in pos], fin_real / 1e6, 'o-', color='#2c3e50',
        lw=1.8, ms=4, label='Resultado Financiero', zorder=3)
ax.axhline(0, color='black', lw=1.0)
if MES_INICIO in idx:
    ax.axvline(list(idx).index(MES_INICIO), color='#e67e22', lw=1.5, ls='--', alpha=0.7, label='Inicio Milei (Dic-2023)')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.set_ylabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.25, zorder=0)
fmt_int(ax, idx)
plt.tight_layout()
save_fig(fig, '01_resultado_primario_financiero')
plt.show()

# Tabla anual: usar el indice de 'primario' (DatetimeIndex) para agrupar por año
df_anual = pd.DataFrame({'Primario real (B)': prim_real/1e6, 'Financiero real (B)': fin_real/1e6},
                         index=idx)
print(f'Resumen anual - Sector Publico Total (real, {BASE.strftime("%b %Y")}):')
print(df_anual.groupby(df_anual.index.year).sum().round(1).to_string())

In [ ]:
# ── Celda 3: Grafico 01 - Resultado Primario y Financiero ─────────────
primario   = drop_gap(get_serie_total('XIV_RESULTADO_PRIMARIO'))
financiero = drop_gap(get_serie_total('XV_RESULTADO_FINANCIERO'))
prim_real  = deflactar(primario)
fin_real   = deflactar(financiero)
idx = primario.index
pos = list(range(len(idx)))

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
colors = ['#27ae60' if v >= 0 else '#c0392b' for v in prim_real]
ax.bar(pos, prim_real / 1e6, color=colors, alpha=0.9, label='Resultado Primario', zorder=2)
ax.plot([p + 0.5 for p in pos], fin_real / 1e6, 'o-', color='#2c3e50',
        lw=1.8, ms=4, label='Resultado Financiero', zorder=3)
ax.axhline(0, color='black', lw=1.0)
ax.axvline(pos[idx.get_loc(MES_INICIO)] if MES_INICIO in idx else pos[0],
           color='#e67e22', lw=1.5, ls='--', alpha=0.7, label='Inicio Milei (Dic-2023)')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.set_ylabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.25, zorder=0)
fmt_int(ax, idx)
plt.tight_layout()
save_fig(fig, '01_resultado_primario_financiero')
plt.show()

# Tabla anual
df_anual = pd.DataFrame({'Primario real (B)': prim_real/1e6, 'Financiero real (B)': fin_real/1e6})
print(f'Resumen anual - Sector Publico Total (real, {BASE.strftime("%b %Y")}):')
print(df_anual.groupby(df_anual.index.year).sum().round(1).to_string())

In [ ]:
# ── Celda 4: Grafico 02 - Composicion del gasto corriente ────────────
comp_gasto = {
    'II3_PRESTACIONES_SEG_SOCIAL': 'Prestaciones Seg.Social',
    'II2_INTERESES':               'Intereses',
    'II1a_REMUNERACIONES':         'Remuneraciones',
    'II4a_TRANSF_SECTOR_PRIVADO':  'Subsidios',
    'II4b1_TRANSF_PROVINCIAS_CABA':'Transf. Provincias',
    'II4b2_TRANSF_UNIVERSIDADES':  'Universidades',
}
colors_g = ['#c0392b','#8e44ad','#2980b9','#16a085','#6c3483','#d68910']

idx_full = get_serie('II_GASTOS_CORRIENTES').index
idx  = idx_full[idx_full != GAP_DATE]
pos  = list(range(len(idx)))
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
bottom = np.zeros(len(idx))
for (cod, label), color in zip(comp_gasto.items(), colors_g):
    vals = deflactar(get_serie(cod).reindex(idx).fillna(0)) / 1e6
    ax.bar(pos, vals, bottom=bottom, label=label, color=color, alpha=0.88, zorder=2)
    bottom += vals
ax.axvline(pos[list(idx).index(MES_INICIO)] if MES_INICIO in idx else len(pos)-1,
           color='#e67e22', lw=1.5, ls='--', alpha=0.7)
ax.set_ylabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(axis='y', alpha=0.25, zorder=0)
fmt_int(ax, idx)
plt.tight_layout()
save_fig(fig, '02_composicion_gasto')
plt.show()

In [ ]:
# ── Celda 5: Grafico 03 - Composicion de ingresos ────────────────────
idx_full = get_serie('I_INGRESOS_CORRIENTES').index
idx  = idx_full[idx_full != GAP_DATE]
pos  = list(range(len(idx)))
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
bottom = np.zeros(len(idx))
for cod, label, color in [
    ('I1_INGRESOS_TRIBUTARIOS',    'Tributarios',    '#1e8449'),
    ('I2_APORTES_SEG_SOCIAL',      'Seg. Social',    '#1a5276'),
    ('I3_INGRESOS_NO_TRIBUTARIOS', 'No tributarios', '#b7950b'),
]:
    vals = deflactar(get_serie(cod).reindex(idx).fillna(0)) / 1e6
    ax.bar(pos, vals, bottom=bottom, label=label, color=color, alpha=0.88, zorder=2)
    bottom += vals
ax.axvline(pos[list(idx).index(MES_INICIO)] if MES_INICIO in idx else len(pos)-1,
           color='#e67e22', lw=1.5, ls='--', alpha=0.7)
ax.set_ylabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.25, zorder=0)
fmt_int(ax, idx)
plt.tight_layout()
save_fig(fig, '03_composicion_ingresos')
plt.show()

In [ ]:
# ── Celda 6: Grafico 04 - Transferencias a provincias ────────────────
corr_tes = drop_gap(get_serie('II4b1_TRANSF_PROVINCIAS_CABA', 'tesoro_nacional'))
cap_tes  = drop_gap(get_serie('V2a_TRANSF_CAPITAL_PROVINCIAS', 'tesoro_nacional'))
corr_real = pd.Series(deflactar(corr_tes), index=corr_tes.index)
cap_real  = pd.Series(deflactar(cap_tes),  index=cap_tes.index)
idx = corr_real.index
pos = list(range(len(idx)))

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
corr_vals = corr_real.values / 1000
cap_vals  = cap_real.reindex(idx).fillna(0).values / 1000
ax.bar(pos, corr_vals, color='#6c3483', alpha=0.9, label='Corrientes (Tesoro)', zorder=2)
ax.bar(pos, cap_vals, bottom=corr_vals, color='#ca6f1e', alpha=0.9, label='Capital (Tesoro)', zorder=2)
ax.axvline(pos[list(idx).index(MES_INICIO)] if MES_INICIO in idx else len(pos)-1,
           color='#e67e22', lw=1.5, ls='--', alpha=0.7)
ax.set_ylabel(f'Miles de MM ARS ({BASE.strftime("%b %Y")})')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.25, zorder=0)
fmt_int(ax, idx)
plt.tight_layout()
save_fig(fig, '04_transferencias_provincias')
plt.show()

In [ ]:
# ── Celda 7: Grafico 05 - Ajuste por componente AIF (dic-23 → ultimo mes) ──
# Comparacion mes a mes: dic-2023 vs ultimo mes disponible
# Usa AIF Administracion Nacional, valores mensuales en pesos constantes

conceptos_mensual = {
    'I_INGRESOS_CORRIENTES':        'Ingresos corrientes',
    'II_GASTOS_CORRIENTES':         'Gastos corrientes',
    'II2_INTERESES':                '  Intereses',
    'II3_PRESTACIONES_SEG_SOCIAL':  '  Prestaciones Seg.Social',
    'II1a_REMUNERACIONES':          '  Remuneraciones',
    'II4b1_TRANSF_PROVINCIAS_CABA': '  Transf. Provincias/CABA',
    'II4b2_TRANSF_UNIVERSIDADES':   '  Universidades',
    'II4a_TRANSF_SECTOR_PRIVADO':   '  Subsidios (transf. privado)',
    'V_GASTOS_CAPITAL':             'Gastos de capital',
    'XIV_RESULTADO_PRIMARIO':       'RESULTADO PRIMARIO',
}

rows = []
for cod, nombre in conceptos_mensual.items():
    s = get_serie(cod)
    ipc_arr = ipc.reindex(s.index.to_period('M').to_timestamp()).values
    s_real  = pd.Series(s.values * (IPC_BASE / ipc_arr), index=s.index)
    v_ini   = s_real.get(MES_INICIO)
    v_fin   = s_real.get(MES_FINAL)
    if v_ini is None or v_fin is None or np.isnan(v_ini) or np.isnan(v_fin):
        continue
    var = (v_fin - v_ini) / 1e6
    pct = (v_fin / v_ini - 1) * 100 if v_ini != 0 else 0
    rows.append({'Concepto': nombre, 'Dic-23 (B)': round(v_ini/1e6,1),
                 f'{MES_FINAL.strftime("%b-%y")} (B)': round(v_fin/1e6,1),
                 'Variacion (B)': round(var,1), 'Var %': f'{pct:+.1f}%'})

df_comp = pd.DataFrame(rows)
print(f'Variacion mensual real: {MES_INICIO.strftime("%b-%Y")} → {MES_FINAL.strftime("%b-%Y")}')
print(df_comp.to_string(index=False))

# Grafico barras horizontales
mask_det = df_comp['Concepto'].str.startswith('  ')
df_det   = df_comp[mask_det].copy()
df_det['Concepto'] = df_det['Concepto'].str.strip()
df_det = df_det.sort_values('Variacion (B)')

fig, ax = plt.subplots(figsize=(13, 6))
colors = ['#922b21' if v < 0 else '#1a5276' for v in df_det['Variacion (B)']]
bars   = ax.barh(df_det['Concepto'], df_det['Variacion (B)'], color=colors,
                 alpha=0.92, edgecolor='white', linewidth=0.5, height=0.65)
ax.axvline(0, color='#2c3e50', lw=1.2)
for bar, row in zip(bars, df_det.itertuples()):
    val = row._4  # Variacion (B)
    pct = row._5  # Var %
    offset = 0.2 if val >= 0 else -0.2
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:+.1f} B  ({pct})', va='center',
            ha='left' if val >= 0 else 'right', fontsize=8.5, fontweight='bold')
ax.set_xlabel(f'Variacion real {MES_INICIO.strftime("%b-%Y")} → {MES_FINAL.strftime("%b-%Y")} | Billones ARS ({BASE.strftime("%b %Y")})')
ax.set_xlim(df_det['Variacion (B)'].min() - 2, df_det['Variacion (B)'].max() + 4)
ax.grid(axis='x', alpha=0.2)
ax.spines['left'].set_visible(False)
ax.text(0.99, 0.02, 'Rojo = baja  |  Azul = suba',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=8, color='gray', style='italic')
plt.tight_layout()
save_fig(fig, '05_ajuste_componentes_mensual')
plt.show()

In [ ]:
# ── Celda 8: Graficos 06 y 07 - Torta y barras por rubro IMIG ────────
# Comparacion: dic-2023 vs ultimo mes disponible en IMIG
# Rubros funcionales (concepto_codigo, nivel)

rubros = [
    ('Obra publica',           'Gastos_capital',               1),
    ('Subsidios',              'Subsidios_economicos',          1),
    ('Transf. Provincias',     'Transf_corrientes_provincias',  1),
    ('Salarios',               'Salarios',                      2),
    ('Jubilaciones',           'Jubilaciones_pensiones',         2),
    ('Universidades',          'Transf_universidades',           1),
    ('Pensiones NC',           'Pensiones_no_contributivas',     2),
    ('PAMI',                   'INSSJP_PAMI',                    2),
    ('Otros prog. sociales',   'Otros_prog_sociales',            2),
    ('AUH',                    'AUH',                            2),
]

# Calcular variacion dic-2023 → ultimo mes disponible
MES_IMIG_FIN = df_imig['fecha'].max()

filas_imig = []
for nombre, cod, nivel in rubros:
    v_ini = get_imig_mes(cod, MES_INICIO, nivel)
    v_fin = get_imig_mes(cod, MES_IMIG_FIN, nivel)
    if v_ini is None or v_fin is None:
        continue
    var   = v_fin - v_ini
    pct   = (v_fin / v_ini - 1) * 100 if v_ini != 0 else 0
    filas_imig.append({'Rubro': nombre, 'Dic-23 (B)': round(v_ini,1),
                       f'{MES_IMIG_FIN.strftime("%b-%y")} (B)': round(v_fin,1),
                       'Variacion (B)': round(var,1), 'Var %': f'{pct:+.1f}%'})

df_imig_rubros = pd.DataFrame(filas_imig)
print(f'Variacion IMIG: {MES_INICIO.strftime("%b-%Y")} → {MES_IMIG_FIN.strftime("%b-%Y")}')
print(df_imig_rubros.to_string(index=False))

# ── Grafico 06: Torta con % de la BAJA (solo rubros con reduccion) ────
df_baja = df_imig_rubros[df_imig_rubros['Variacion (B)'] < 0].copy()
df_baja['abs_var'] = df_baja['Variacion (B)'].abs()
total_baja = df_baja['abs_var'].sum()

colors_pie = ['#922b21','#1a5276','#6c3483','#1e8449','#784212',
              '#0e6655','#4d5656','#2e4053','#7d6608']
fig, ax = plt.subplots(figsize=(10, 8))
wedges, texts, autotexts = ax.pie(
    df_baja['abs_var'], labels=df_baja['Rubro'],
    autopct='%1.1f%%', colors=colors_pie[:len(df_baja)],
    startangle=140, pctdistance=0.75, labeldistance=1.12,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.2}
)
for at in autotexts:
    at.set_fontsize(8)
    at.set_fontweight('bold')
    at.set_color('white')
for t in texts:
    t.set_fontsize(9)
ax.set_title(
    f'Distribucion de la baja del gasto\n'
    f'{MES_INICIO.strftime("%b-%Y")} → {MES_IMIG_FIN.strftime("%b-%Y")}\n'
    f'Total baja identificada: {total_baja:.1f} B ARS ({BASE.strftime("%b %Y")})',
    fontsize=11, pad=15
)
plt.tight_layout()
save_fig(fig, '06_torta_recorte_gasto')
plt.show()

# ── Grafico 07: Barras horizontales variacion real por rubro ──────────
df_imig_rubros_sorted = df_imig_rubros.sort_values('Variacion (B)')
colors_bar = ['#922b21' if v < 0 else '#1a5276' for v in df_imig_rubros_sorted['Variacion (B)']]

fig, ax = plt.subplots(figsize=(13, 7))
bars = ax.barh(df_imig_rubros_sorted['Rubro'], df_imig_rubros_sorted['Variacion (B)'],
               color=colors_bar, alpha=0.92, edgecolor='white', linewidth=0.5, height=0.65)
ax.axvline(0, color='#2c3e50', lw=1.2)
for bar, row in zip(bars, df_imig_rubros_sorted.itertuples()):
    val = row._4  # Variacion (B)
    pct = row._5  # Var %
    offset = 0.1 if val >= 0 else -0.1
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:+.1f} B  ({pct})', va='center',
            ha='left' if val >= 0 else 'right', fontsize=9, fontweight='bold')
ax.set_xlabel(f'Variacion real | Billones ARS ({BASE.strftime("%b %Y")})')
ax.set_xlim(df_imig_rubros_sorted['Variacion (B)'].min() - 1,
            df_imig_rubros_sorted['Variacion (B)'].max() + 3)
ax.grid(axis='x', alpha=0.2)
ax.spines['left'].set_visible(False)
ax.set_title(f'Variacion del gasto por rubro: {MES_INICIO.strftime("%b-%Y")} → {MES_IMIG_FIN.strftime("%b-%Y")}\n(Rojo = baja  |  Azul = suba)',
             fontsize=11)
plt.tight_layout()
save_fig(fig, '07_recorte_por_rubro')
plt.show()

In [ ]:
# ── Celda 9: Resumen completo del ajuste fiscal ───────────────────────
# PIB nominal aproximado (billones ARS)
PIB_B = {2020:44.9, 2021:72.0, 2022:115.0,
         2023:143.2, 2024:586.7, 2025:725.0}
# Fuente: 2023=deficit 8.74B/6.1%PIB; 2024=superavit 1.76B/0.3%; 2025=1.45B/0.2% (Hacienda)

print('=' * 80)
print('RESUMEN DEL AJUSTE FISCAL - GESTION MILEI')
print(f'Periodo: {MES_INICIO.strftime("%b-%Y")} → {MES_FINAL.strftime("%b-%Y")}')
print(f'Valores en pesos constantes de {BASE.strftime("%b-%Y")} (billones ARS = 10^12 pesos)')
print('=' * 80)

# 1. Resultado fiscal anual
df_macro = pd.DataFrame()
for concepto, col in [
    ('XIV_RESULTADO_PRIMARIO',               'Prim_real'),
    ('XV_RESULTADO_FINANCIERO',              'Fin_real'),
    ('XII_GASTOS_PRIMARIOS_DESPUES_FIGURAT', 'Gasto_real'),
    ('I_INGRESOS_CORRIENTES',                'Ing_real'),
    ('II2_INTERESES',                        'Int_real'),
]:
    s = get_serie_total(concepto)
    r = pd.Series(deflactar(s), index=s.index)
    df_macro[col] = r.groupby(r.index.year).sum() / 1e6

for concepto, col in [('XIV_RESULTADO_PRIMARIO','Prim_nom'),('XV_RESULTADO_FINANCIERO','Fin_nom')]:
    s = get_serie_total(concepto)
    df_macro[col] = s.groupby(s.index.year).sum() / 1e6

df_macro['Prim%PIB'] = df_macro.apply(
    lambda r: f"{r['Prim_nom']/PIB_B[r.name]*100:+.1f}%" if r.name in PIB_B else 'n/d', axis=1)
df_macro['Fin%PIB']  = df_macro.apply(
    lambda r: f"{r['Fin_nom']/PIB_B[r.name]*100:+.1f}%" if r.name in PIB_B else 'n/d', axis=1)

print()
print('1. RESULTADO FISCAL ANUAL - SECTOR PUBLICO TOTAL')
t1 = df_macro[['Prim_real','Fin_real','Prim%PIB','Fin%PIB','Ing_real','Gasto_real','Int_real']]
t1 = t1.rename(columns={'Prim_real':'Primario(B)','Fin_real':'Financiero(B)',
                          'Prim%PIB':'Prim%PIB','Fin%PIB':'Fin%PIB',
                          'Ing_real':'Ingresos(B)','Gasto_real':'Gasto prim.(B)','Int_real':'Intereses(B)'})
print(t1[t1.index.isin(range(2020,2027))].round(1).to_string())
print('  * PIB aproximado | Fuentes: INDEC, Hacienda')

# 2. Magnitud del ajuste
gp23 = df_macro.loc[2023,'Gasto_real']
gp24 = df_macro.loc[2024,'Gasto_real']
gp25 = df_macro.loc[2025,'Gasto_real']
rp23 = df_macro.loc[2023,'Prim_real']
rp24 = df_macro.loc[2024,'Prim_real']
rp25 = df_macro.loc[2025,'Prim_real']

print()
print('2. MAGNITUD DEL AJUSTE (variacion de año completo 2023 → 2024/2025)')
print(f'   Gasto prim. 2023: {gp23:.1f} B | 2024: {gp24:.1f} B ({(gp24-gp23)/gp23*100:+.1f}%) | 2025: {gp25:.1f} B ({(gp25-gp23)/gp23*100:+.1f}%)')
print(f'   Resultado primario 2023: {rp23:+.1f} B | 2024: {rp24:+.1f} B | 2025: {rp25:+.1f} B')
print(f'   Mejora resultado primario 2023→2024: {rp24-rp23:+.1f} B')

# 3. Variacion mensual dic-2023 → ultimo mes AIF
print()
print(f'3. VARIACION MENSUAL {MES_INICIO.strftime("%b-%Y").upper()} → {MES_FINAL.strftime("%b-%Y").upper()} (AIF, Adm. Nacional)')
print(df_comp[['Concepto','Dic-23 (B)',f'{MES_FINAL.strftime("%b-%y")} (B)','Variacion (B)','Var %']].to_string(index=False))

# 4. Desglose funcional IMIG
print()
print(f'4. DESGLOSE FUNCIONAL POR RUBRO (IMIG, {MES_INICIO.strftime("%b-%Y")} → {MES_IMIG_FIN.strftime("%b-%Y")})')
print(df_imig_rubros.to_string(index=False))
print()
baja_total = df_imig_rubros[df_imig_rubros['Variacion (B)'] < 0]['Variacion (B)'].sum()
suba_total = df_imig_rubros[df_imig_rubros['Variacion (B)'] > 0]['Variacion (B)'].sum()
print(f'   Rubros con baja: {baja_total:.1f} B  |  Rubros con suba: {suba_total:+.1f} B  |  Neto: {baja_total+suba_total:.1f} B')
print()
print('   Nota metodologica:')
print('   - Comparacion mensual (dic-23 vs ultimo mes) puede ser afectada por estacionalidad')
print('   - AIF = base caja, IMIG = consolidado sector publico; perimetros distintos')
print('   - PIB nominal es aproximado; usar para ordenes de magnitud')

In [ ]:
# ── Celda 10: Exportar Excel + ZIP ────────────────────────────────────
EXCEL_FILE = 'analisis_fiscal_resultados.xlsx'
ZIP_FILE   = 'analisis_fiscal.zip'

# Serie mensual
idx_base = get_serie_total('XIV_RESULTADO_PRIMARIO').index
rows_m   = {'fecha': idx_base}
for nombre, cod in [
    ('ingresos_corrientes',  'I_INGRESOS_CORRIENTES'),
    ('gastos_corrientes',    'II_GASTOS_CORRIENTES'),
    ('intereses',            'II2_INTERESES'),
    ('prestaciones',         'II3_PRESTACIONES_SEG_SOCIAL'),
    ('resultado_primario',   'XIV_RESULTADO_PRIMARIO'),
    ('resultado_financiero', 'XV_RESULTADO_FINANCIERO'),
]:
    s = get_serie_total(cod)
    rows_m[f'{nombre}_nominal_MM'] = s.reindex(idx_base).values
    rows_m[f'{nombre}_real_MM']    = deflactar(s.reindex(idx_base))
df_serie = pd.DataFrame(rows_m)

# Resumen anual
df_anual_exp = df_serie.copy()
df_anual_exp['anio'] = pd.to_datetime(df_anual_exp['fecha']).dt.year
df_anual_exp = df_anual_exp.groupby('anio').sum(numeric_only=True)

# Transferencias
prov_rows = []
for cod, tipo in [('II4b1_TRANSF_PROVINCIAS_CABA','Corrientes'),('V2a_TRANSF_CAPITAL_PROVINCIAS','Capital')]:
    for ss in ['tesoro_nacional','total_adm_nacional']:
        s = get_serie(cod, ss)
        if len(s) == 0: continue
        prov_rows.append(pd.DataFrame({'fecha':s.index,'tipo':tipo,'subsector':ss,
                                        'nominal_MM':s.values,'real_MM':deflactar(s)}))
df_prov_exp = pd.concat(prov_rows).sort_values(['fecha','tipo'])

with pd.ExcelWriter(EXCEL_FILE, engine='openpyxl') as writer:
    df_serie.to_excel(writer,         sheet_name='Serie_mensual',       index=False)
    df_anual_exp.to_excel(writer,     sheet_name='Resumen_anual',       index=True)
    df_prov_exp.to_excel(writer,      sheet_name='Transferencias_prov', index=False)
    df_comp.to_excel(writer,          sheet_name='Ajuste_AIF_mensual',  index=False)
    df_imig_rubros.to_excel(writer,   sheet_name='Ajuste_IMIG_rubros',  index=False)

print(f'Excel: {EXCEL_FILE}')
print('Hojas: Serie_mensual | Resumen_anual | Transferencias_prov | Ajuste_AIF_mensual | Ajuste_IMIG_rubros')

with zipfile.ZipFile(ZIP_FILE, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(EXCEL_FILE)
    for g in GRAFICOS:
        if os.path.exists(g):
            zf.write(g)

print(f'\nZIP: {ZIP_FILE}')
print(f'Contenido: Excel + {len([g for g in GRAFICOS if os.path.exists(g)])} graficos')
for g in GRAFICOS:
    if os.path.exists(g):
        print(f'  {g}  ({os.path.getsize(g)//1024} KB)')

if IN_COLAB:
    from google.colab import files
    files.download(ZIP_FILE)
    print('\nDescarga iniciada')
else:
    print(f'\nGuardado en: {os.path.abspath(ZIP_FILE)}')